# Week 4: Agribusiness SQL Analysis and Visualization
**Google Colab notebook | GreenField Agro**

This notebook uses SQLite SQL queries to analyse farm, crop and seasonal performance, then visualizes the SQL results. All sample data is synthetic and for coursework demonstration only.

## 1. Connect to SQLite and prepare libraries

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

conn = sqlite3.connect(":memory:")
print("SQLite database ready")

## 2. Create tables

In [ ]:
conn.executescript('''
CREATE TABLE farms (
  farm_id INTEGER PRIMARY KEY,
  farm_name TEXT NOT NULL,
  location TEXT
);
CREATE TABLE crops (
  crop_id INTEGER PRIMARY KEY,
  crop_name TEXT NOT NULL,
  crop_category TEXT
);
CREATE TABLE production (
  record_id INTEGER PRIMARY KEY,
  farm_id INTEGER NOT NULL,
  crop_id INTEGER NOT NULL,
  season TEXT NOT NULL,
  area_acres REAL NOT NULL,
  production_tonnes REAL NOT NULL,
  operating_cost_inr REAL NOT NULL,
  revenue_inr REAL NOT NULL,
  FOREIGN KEY (farm_id) REFERENCES farms(farm_id),
  FOREIGN KEY (crop_id) REFERENCES crops(crop_id)
);
''')
print("Tables created successfully")

## 3. Insert sample data (36 records)

In [ ]:
farms = [
    (1, "North Field", "North Zone"),
    (2, "River Farm", "River Zone"),
    (3, "East Block", "East Zone")
]
crops = [(1, "Rice", "Cereal"), (2, "Maize", "Cereal"), (3, "Potato", "Vegetable")]
seasons = ["2024-Kharif", "2024-Rabi", "2025-Kharif", "2025-Rabi"]

conn.executemany("INSERT INTO farms VALUES (?, ?, ?)", farms)
conn.executemany("INSERT INTO crops VALUES (?, ?, ?)", crops)

# Synthetic teaching values: yield (tonnes/acre), cost (INR/acre), revenue (INR/acre)
base = {
    1: (9.0, 52000, 145000),
    2: (7.0, 41000, 112000),
    3: (12.0, 85000, 230000)
}
farm_factor = {1: 1.00, 2: 1.08, 3: 0.94}
season_factor = [0.96, 1.02, 1.04, 0.99]
rows = []
rid = 1
for farm_id in (1, 2, 3):
    for crop_id in (1, 2, 3):
        y, cost_per_acre, revenue_per_acre = base[crop_id]
        for si, season in enumerate(seasons):
            area = 10 + ((farm_id * 3 + crop_id * 2 + si) % 7)
            factor = farm_factor[farm_id] * season_factor[si]
            production = area * y * factor
            cost = area * cost_per_acre * (1 + 0.015 * si)
            revenue = area * revenue_per_acre * factor
            rows.append((rid, farm_id, crop_id, season, area,
                         round(production, 2), round(cost, 2), round(revenue, 2)))
            rid += 1

conn.executemany("INSERT INTO production VALUES (?, ?, ?, ?, ?, ?, ?, ?)", rows)
conn.commit()
print("Inserted rows:", len(rows))
display(pd.read_sql_query("SELECT * FROM production LIMIT 5", conn))

## 4. SQL Query 1 — Data quality checks

In [ ]:
q1 = '''
SELECT COUNT(*) AS total_records,
       SUM(CASE WHEN area_acres IS NULL OR production_tonnes IS NULL
                     OR operating_cost_inr IS NULL OR revenue_inr IS NULL
                THEN 1 ELSE 0 END) AS missing_value_records,
       SUM(CASE WHEN area_acres <= 0 OR production_tonnes < 0
                     OR operating_cost_inr < 0 OR revenue_inr < 0
                THEN 1 ELSE 0 END) AS invalid_value_records
FROM production;
'''
display(pd.read_sql_query(q1, conn))

## 5. SQL Query 2 — Crop performance KPIs
Weighted yield = total production / total area. Cost per kg converts tonnes to kilograms.

In [ ]:
q2 = '''
SELECT c.crop_name,
       ROUND(SUM(p.area_acres), 2) AS area_acres,
       ROUND(SUM(p.production_tonnes), 2) AS production_tonnes,
       ROUND(SUM(p.production_tonnes) / SUM(p.area_acres), 2) AS yield_tonnes_per_acre,
       ROUND(SUM(p.operating_cost_inr) / (SUM(p.production_tonnes) * 1000), 2) AS cost_per_kg_inr,
       ROUND(SUM(p.revenue_inr), 2) AS revenue_inr,
       ROUND(SUM(p.revenue_inr) - SUM(p.operating_cost_inr), 2) AS operating_profit_inr,
       ROUND(100.0 * (SUM(p.revenue_inr) - SUM(p.operating_cost_inr))
             / NULLIF(SUM(p.revenue_inr), 0), 2) AS operating_margin_pct
FROM production p JOIN crops c ON p.crop_id = c.crop_id
GROUP BY c.crop_name ORDER BY operating_profit_inr DESC;
'''
crop_summary = pd.read_sql_query(q2, conn)
display(crop_summary)

## 6. SQL Query 3 — Seasonal yield trend

In [ ]:
q3 = '''
SELECT season,
       ROUND(SUM(area_acres), 2) AS area_acres,
       ROUND(SUM(production_tonnes), 2) AS production_tonnes,
       ROUND(SUM(production_tonnes) / SUM(area_acres), 2) AS yield_tonnes_per_acre
FROM production GROUP BY season ORDER BY season;
'''
season_summary = pd.read_sql_query(q3, conn)
display(season_summary)

## 7. SQL Query 4 — Farm efficiency by crop

In [ ]:
q4 = '''
SELECT f.farm_name, c.crop_name,
       ROUND(SUM(p.area_acres), 2) AS area_acres,
       ROUND(SUM(p.production_tonnes), 2) AS production_tonnes,
       ROUND(SUM(p.production_tonnes) / SUM(p.area_acres), 2) AS yield_tonnes_per_acre,
       ROUND(SUM(p.operating_cost_inr) / (SUM(p.production_tonnes) * 1000), 2) AS cost_per_kg_inr
FROM production p
JOIN farms f ON p.farm_id = f.farm_id
JOIN crops c ON p.crop_id = c.crop_id
GROUP BY f.farm_name, c.crop_name
ORDER BY f.farm_name, yield_tonnes_per_acre DESC;
'''
farm_efficiency = pd.read_sql_query(q4, conn)
display(farm_efficiency)

## 8. SQL Query 5 — Operating cost by crop
The sample schema contains total operating cost, not separate seed/fertilizer/labour categories.

In [ ]:
q5 = '''
SELECT c.crop_name, ROUND(SUM(p.operating_cost_inr), 2) AS operating_cost_inr
FROM production p JOIN crops c ON p.crop_id = c.crop_id
GROUP BY c.crop_name ORDER BY operating_cost_inr DESC;
'''
cost_summary = pd.read_sql_query(q5, conn)
display(cost_summary)

## 9. SQL Query 6 — Crop-season profitability

In [ ]:
q6 = '''
SELECT c.crop_name, p.season,
       ROUND(SUM(p.revenue_inr), 2) AS revenue_inr,
       ROUND(SUM(p.operating_cost_inr), 2) AS operating_cost_inr,
       ROUND(SUM(p.revenue_inr) - SUM(p.operating_cost_inr), 2) AS operating_profit_inr
FROM production p JOIN crops c ON p.crop_id = c.crop_id
GROUP BY c.crop_name, p.season
ORDER BY c.crop_name, p.season;
'''
profitability = pd.read_sql_query(q6, conn)
display(profitability)

## 10. SQL Query 7 — Rank seasonal yield within each crop
This uses the SQL window function `RANK()`.

In [ ]:
q7 = '''
WITH seasonal_yield AS (
  SELECT c.crop_name, p.season,
         SUM(p.production_tonnes) / SUM(p.area_acres) AS yield_tonnes_per_acre
  FROM production p JOIN crops c ON p.crop_id = c.crop_id
  GROUP BY c.crop_name, p.season
)
SELECT crop_name, season, ROUND(yield_tonnes_per_acre, 2) AS yield_tonnes_per_acre,
       RANK() OVER (PARTITION BY crop_name ORDER BY yield_tonnes_per_acre DESC) AS yield_rank
FROM seasonal_yield
ORDER BY crop_name, yield_rank;
'''
yield_ranking = pd.read_sql_query(q7, conn)
display(yield_ranking)

## 11. SQL Query 8 — Overall management dashboard

In [ ]:
q8 = '''
SELECT COUNT(*) AS production_records,
       ROUND(SUM(area_acres), 2) AS total_area_acres,
       ROUND(SUM(production_tonnes), 2) AS total_production_tonnes,
       ROUND(SUM(production_tonnes) / SUM(area_acres), 2) AS overall_yield_tonnes_per_acre,
       ROUND(SUM(revenue_inr), 2) AS total_revenue_inr,
       ROUND(SUM(operating_cost_inr), 2) AS total_operating_cost_inr,
       ROUND(SUM(revenue_inr) - SUM(operating_cost_inr), 2) AS operating_profit_inr,
       ROUND(100.0 * (SUM(revenue_inr) - SUM(operating_cost_inr))
             / NULLIF(SUM(revenue_inr), 0), 2) AS operating_margin_pct
FROM production;
'''
dashboard = pd.read_sql_query(q8, conn)
display(dashboard)

## 12. Visualizations
These charts are generated from the SQL query outputs above.

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(crop_summary["crop_name"], crop_summary["yield_tonnes_per_acre"])
plt.title("Weighted Yield by Crop")
plt.xlabel("Crop")
plt.ylabel("Yield (tonnes per acre)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 5))
plt.plot(season_summary["season"], season_summary["yield_tonnes_per_acre"], marker="o")
plt.title("Seasonal Yield Trend")
plt.xlabel("Season")
plt.ylabel("Yield (tonnes per acre)")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.barh(cost_summary["crop_name"], cost_summary["operating_cost_inr"])
plt.title("Total Operating Cost by Crop")
plt.xlabel("Operating cost (INR)")
plt.ylabel("Crop")
plt.tight_layout()
plt.show()

## 13. Interpretation and business recommendations

**How to interpret the outputs**
- Compare crops using weighted yield, cost per kg, operating profit and operating margin together.
- Use the seasonal line chart to identify periods where yield rises or falls.
- Compare the same crop across farms to identify practices that may be worth investigating or sharing.
- Total operating cost depends on cultivated area; cost per kg is more useful for comparing production efficiency.
- Operating profit here is revenue minus recorded operating cost. It excludes any overheads not present in the dataset.

**Why these visualizations?**
- Bar chart: compares discrete crop categories clearly.
- Line chart: shows the ordered seasonal trend.
- Horizontal bar chart: makes cost differences and crop labels easy to read.
- Tables: retain exact values for operational review.

**Recommendations**
1. Prioritize crops using profitability and cost per kg, not yield alone.
2. Investigate high-performing farm-crop combinations and document their cultivation practices.
3. Review low-yield seasons for possible weather, irrigation, input or scheduling issues.
4. Record detailed input costs (seed, fertilizer, labour, irrigation, machinery and transport) to enable true cost-composition analysis.
5. Update the database after every season and monitor yield, cost per kg, revenue and operating margin.

**Limitations:** All records are synthetic. Results are illustrative and must not be treated as actual farm performance. Real decisions require verified farm records, local agronomic knowledge and current market prices.

## Conclusion
SQL makes agribusiness reporting repeatable and auditable. Combining query outputs with suitable charts helps stakeholders monitor productivity, seasonal trends, cost efficiency and profitability.